# NB05 — Statistical Inference: t-stats, p-values, Confidence Intervals

> **How do we know if a coefficient is "real" and not just noise?**


## 1. Standard Error of a coefficient

β̂₁ is an estimate — it would be different if we collected new data.
The **Standard Error (SE)** measures how much it would vary:

```
SE(β̂₁) = √[ σ² / Σ(xᵢ−x̄)² ]

where σ² = SS_res / (n−2)   ← residual variance (unbiased, divides by n−2)
```

Larger SE → more uncertainty → less trustworthy estimate.


In [ ]:
import numpy as np
import scipy.stats as stats

np.random.seed(7)
n = 30
X = np.linspace(1, 10, n)
true_slope = 2.5
y = true_slope * X + 5 + np.random.normal(0, 4, n)

xbar = X.mean()
ybar = y.mean()

b1 = np.sum((X-xbar)*(y-ybar)) / np.sum((X-xbar)**2)
b0 = ybar - b1*xbar

y_hat  = b0 + b1*X
resid  = y - y_hat
sigma2 = np.sum(resid**2) / (n-2)     # unbiased residual variance
sxx    = np.sum((X-xbar)**2)

se_b1 = np.sqrt(sigma2 / sxx)
se_b0 = np.sqrt(sigma2 * (1/n + xbar**2/sxx))

print(f"True slope:      {true_slope}")
print(f"Estimated slope: {b1:.4f}")
print(f"SE(slope):       {se_b1:.4f}")
print(f"SE(intercept):   {se_b0:.4f}")


## 2. t-statistic

```
t = β̂ / SE(β̂)
```

Measures: how many standard errors is our estimate away from **zero**?

Under H₀: β = 0, this follows a **t-distribution with n−2 degrees of freedom**.

Rule of thumb: |t| > 2 is roughly significant at the 5% level.


In [ ]:
# t-statistics and p-values
df = n - 2   # degrees of freedom

t_b0 = b0 / se_b0
t_b1 = b1 / se_b1

# Two-sided p-value: P(|T| > |t_obs|) under H₀
p_b0 = 2 * stats.t.sf(abs(t_b0), df)
p_b1 = 2 * stats.t.sf(abs(t_b1), df)

print(f"{'':12}{'Coef':>10}  {'SE':>10}  {'t':>10}  {'p-value':>12}")
print("-"*58)
print(f"{'intercept':12}{b0:>10.4f}  {se_b0:>10.4f}  {t_b0:>10.4f}  {p_b0:>12.6f}")
print(f"{'slope':12}{b1:>10.4f}  {se_b1:>10.4f}  {t_b1:>10.4f}  {p_b1:>12.6f}")

print(f"\np_slope = {p_b1:.6f}")
if p_b1 < 0.05:
    print("Slope is statistically significant at 5% level (p < 0.05)")
else:
    print("Slope is NOT statistically significant at 5% level")


## 3. Confidence Intervals

```
CI₉₅ = β̂ ± t*(df, 0.025) × SE(β̂)
```

The critical value `t*` = 1.96 for large n (comes from the normal approximation).
For small samples, use the exact t-table value.

**Interpretation:** if we repeated the experiment 100 times, ~95 of the resulting CIs would contain the true β.
*(Not: "95% probability the true β is in this interval.")*


In [ ]:
import numpy as np, scipy.stats as stats, matplotlib.pyplot as plt

t_crit = stats.t.ppf(0.975, df)   # t*(n-2, 0.025) for 95% CI
print(f"t* (critical value, df={df}): {t_crit:.4f}  (≈1.96 for large n)")

ci_b0 = (b0 - t_crit*se_b0, b0 + t_crit*se_b0)
ci_b1 = (b1 - t_crit*se_b1, b1 + t_crit*se_b1)

print(f"\n95% CI for intercept: [{ci_b0[0]:.3f}, {ci_b0[1]:.3f}]")
print(f"95% CI for slope:      [{ci_b1[0]:.3f}, {ci_b1[1]:.3f}]")
print(f"True slope {true_slope} inside CI: {ci_b1[0] < true_slope < ci_b1[1]}")

# Visualise: coefficient plot with error bars
fig, ax = plt.subplots(figsize=(6,3))
coefs  = [b0, b1]
ses    = [se_b0, se_b1]
labels = ['Intercept', 'Slope']
ax.errorbar([0,1], coefs, yerr=[t_crit*s for s in ses],
            fmt='o', capsize=8, capthick=2, color='steelblue', markersize=8)
ax.axhline(0, color='red', linewidth=0.8, linestyle='--', label='Zero')
ax.set_xticks([0,1]); ax.set_xticklabels(labels)
ax.set_ylabel('Coefficient value'); ax.set_title('Coefficients ± 95% CI')
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()


In [ ]:
# Verify with statsmodels — shows the full regression table
import statsmodels.api as sm
import numpy as np

X_sm = sm.add_constant(X)    # adds the intercept column
result = sm.OLS(y, X_sm).fit()
print(result.summary())


## 4. Connection: p-value ↔ confidence interval

These two are equivalent:
- p < 0.05   ↔   95% CI does not include 0
- p < 0.01   ↔   99% CI does not include 0

| p-value | Interpretation |
|---------|---------------|
| p < 0.001 | Very strong evidence against H₀ |
| p < 0.01  | Strong evidence |
| p < 0.05  | Conventional threshold — "statistically significant" |
| p ≥ 0.05  | Insufficient evidence to reject H₀ |
| p = 0.04  | Not "almost significant" — just above the threshold means just above |

**Warning:** p < 0.05 does NOT mean:
- The effect is large (check the coefficient size)
- The model is correct
- The result will replicate


## Key Takeaways

| Term | Formula | Meaning |
|------|---------|---------|
| SE | √(σ²/Sxx) | Uncertainty of the estimate |
| t-stat | β̂/SE | Signal-to-noise ratio |
| p-value | 2·P(T>|t|) | Probability of seeing this result if H₀ true |
| 95% CI | β̂ ± t*·SE | Range of plausible coefficient values |

**Next:** NB06 — the four OLS assumptions (LINE) and what happens when they break.
